# بعت الـ CV لكل الإيميلات - من غير ما تنزلي حاجة

شغّلي كل خلية بالترتيب من فوق لتحت (اضغطي على السهم اللي بيظهر يسار كل خلية، أو Shift+Enter).

**قبل ما تبدئي محتاجة:**
1. App Password من Gmail (مش الباسورد العادي) - من myaccount.google.com/security → فعّلي 2-Step Verification → App passwords
2. ملف الإكسل `job_emails_tracker.xlsx`
3. ملف الـ CV `Israa_Daif_CV_Engineering.docx`


### 1) ارفعي الملفين (الإكسل + الـ CV)
هيفتحلك نافذة اختيار ملفات - اختاري الاتنين مع بعض.

In [ ]:
from google.colab import files
uploaded = files.upload()
print("اتحمل:", list(uploaded.keys()))


### 2) ثبتي المكتبة المطلوبة

In [ ]:
!pip install -q openpyxl


### 3) اكتبي إيميلك والـ App Password
هيتسألك تكتبيهم في مكان مخفي (مش هيظهروا على الشاشة) - آمن، Colab مش بيحفظهم.

In [ ]:
from getpass import getpass

GMAIL_ADDRESS = input("إيميلك في Gmail: ").strip()
GMAIL_APP_PASSWORD = getpass("App Password (16 حرف): ").strip()
EXCEL_FILENAME = "job_emails_tracker.xlsx"      # عدّلي لو اسم الملف مختلف
CV_FILENAME = "Israa_Daif_CV_Engineering.docx"  # عدّلي لو اسم الملف مختلف


### 4) نص الإيميل (ممكن تعدليه هنا لو حبيتي)

In [ ]:
EMAIL_SUBJECT = "AI-Driven Telecom Engineer | CV Attached \u2013 Israa Ayman Daif"
EMAIL_BODY = """Dear Hiring Manager,

I hope this email finds you well.

My name is Israa Ayman Daif, a Telecommunications & AI Engineer based in the UAE with a B.Sc. in Communications and Electronics Engineering. I am currently working on telecommunications and electrical drawings (AutoCAD) and supporting AI-driven automation projects at a consulting engineering firm.

I am not applying to a specific vacancy, but I would like to share my CV in case there are any current or upcoming openings at your organization that match my background. I would be glad to have my profile kept on file for future consideration.

Please find my CV attached. I would be happy to provide any further information you may need.

Thank you for your time and consideration.

Best regards,
Israa Ayman Daif
+971 52 491 0964
Israaaymandaif@gmail.com
linkedin.com/in/israa-ayman-daif
"""

DELAY_SECONDS_BETWEEN_EMAILS = 20   # سيبيها بين 15-30 ثانية عشان Gmail متعتبركيش spam
MAX_EMAILS_PER_RUN = 40             # اقصى عدد تبعتيه في المرة الواحدة


### 5) خلية الإرسال - شغليها وسيبيها تشتغل

In [ ]:
import smtplib, time
from email.message import EmailMessage
from pathlib import Path
import openpyxl

excel_path = Path(EXCEL_FILENAME)
cv_path = Path(CV_FILENAME)
cv_bytes = cv_path.read_bytes()

wb = openpyxl.load_workbook(excel_path)
ws = wb.active

def load_rows(ws):
    rows = []
    for r in range(2, ws.max_row + 1):
        email = ws.cell(row=r, column=2).value   # عمود B: الإيميل
        status = ws.cell(row=r, column=5).value  # عمود E: تم الإرسال؟
        if email:
            rows.append((r, str(email).strip(), status))
    return rows

def send_one(smtp, to_addr):
    msg = EmailMessage()
    msg["Subject"] = EMAIL_SUBJECT
    msg["From"] = GMAIL_ADDRESS
    msg["To"] = to_addr
    msg.set_content(EMAIL_BODY)
    msg.add_attachment(
        cv_bytes,
        maintype="application",
        subtype="vnd.openxmlformats-officedocument.wordprocessingml.document",
        filename=cv_path.name,
    )
    smtp.send_message(msg)

rows = load_rows(ws)
to_send = [(r, email) for (r, email, status) in rows if status in (None, "", "\u0644\u0623")]
to_send = to_send[:MAX_EMAILS_PER_RUN]

print(f"هيتبعت {len(to_send)} إيميل دلوقتي (من إجمالي {len(rows)} في الملف).")

with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:
    smtp.login(GMAIL_ADDRESS, GMAIL_APP_PASSWORD)
    for i, (row_idx, email) in enumerate(to_send, start=1):
        try:
            send_one(smtp, email)
            ws.cell(row=row_idx, column=5, value="\u0627\u062a\u0628\u0639\u062a")
            ws.cell(row=row_idx, column=6, value=time.strftime("%Y-%m-%d"))
            print(f"[{i}/{len(to_send)}] اتبعت لـ {email}")
        except Exception as e:
            ws.cell(row=row_idx, column=7, value=f"فشل الإرسال: {e}")
            print(f"[{i}/{len(to_send)}] فشل الإرسال لـ {email}: {e}")
        wb.save(excel_path)
        if i < len(to_send):
            time.sleep(DELAY_SECONDS_BETWEEN_EMAILS)

print("خلصت.")


### 6) نزّلي الإكسل المحدّث على جهازك (فيه الحالة الجديدة لكل صف)

In [ ]:
from google.colab import files
files.download(EXCEL_FILENAME)
